<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. Centralized Data Processing with External Analytics</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 3.3 Lab Iceberg Read Access on UC Managed Tables

In this short lab you take a small slice of the TPC-DS `item` dimension, enable Iceberg V3 reads on the Unity Catalog managed table, observe how deletion vectors work under the covers, then create a managed Iceberg copy and a Delta + UniForm copy of the same data. The PyIceberg step at the end confirms which UC table types are visible to external Iceberg clients (Snowflake, Trino, Flink, EMR, ...).

## Learning Objectives

By the end of this lab, you will be able to:
- Enable **Iceberg V3 reads** on an existing UC managed table and identify what V3 brings (deletion vectors, row tracking)
- Observe how a row-level `DELETE` under V3 records a **deletion vector** rather than rewriting the parquet file
- Create a **managed Iceberg** table and a **Delta + UniForm** table and recognize each by its properties
- Use **PyIceberg** to confirm which UC table types are visible to external Iceberg engines

This lab should take ~15-20 minutes including reading.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li><strong>Serverless Compute, Version 5</strong>: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2272B4">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
<p style="margin: 8px 0 0 0; color: #333">You must have run <strong>0 - Required Setup</strong> first. The preflight in <code>Classroom-Setup-3-lab</code> will tell you if it has not been run.</p>
</div>
</div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-3-lab

## A. Inspect the Baseline UC managed Table

The classroom setup created <code>item_demo</code> as a plain Delta CTAS from <code>samples.tpcds_sf1000.item</code> (~300K rows). It starts out as plain managed Delta with deletion vectors enabled. Confirm what you have.

In [0]:
SHOW TBLPROPERTIES item_demo;

## B. Enable Iceberg V3 Reads

Iceberg V3 read compatibility on <code>item_demo</code> is enabled using an `ALTER TABLE` statement. Deletion vectors and row tracking are enabled automatically in Iceberg V3.

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">V2 vs V3 - Why V3 Is the Default</strong>
            <p style="margin: 8px 0 0 0; color: #333;">All recent Databricks runtimes ship with Iceberg V3 by default and that is what this lab uses. The older Iceberg V2 path is still available for compatibility with engines pinned to an older Iceberg client, but it requires <b>deletion vectors to be disabled</b> before V2 can be enabled, which means row-level <code>DELETE</code>s end up rewriting parquet files. <b>V3 is the recommended path for new tables.</b></p>
        </div>
    </div>
</div>

In [0]:
ALTER TABLE item_demo SET TBLPROPERTIES (
  'delta.columnMapping.mode'             = 'name',
  'delta.enableIcebergCompatV3'          = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
);

SHOW TBLPROPERTIES item_demo;

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #00695c; font-size: 1.1em;">What changed</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The table is still primarily Delta, but Unity Catalog now generates <b>Iceberg V3 metadata alongside the Delta log</b>. External Iceberg clients can read it. Notice <code>delta.enableDeletionVectors=true</code> and <code>delta.enableRowTracking=true</code> - both are kept on, which is what makes the next part interesting.</p>
        </div>
    </div>
</div>

## C. Deletion Vector Behaviour

With V3 read compatibility plus deletion vectors, a row-level <code>DELETE</code> does not rewrite the parquet file - Delta records the deleted rows in a separate <b>deletion vector</b> applied at read time. Run a <code>DELETE</code> and check <code>DESCRIBE HISTORY</code> to see this in action.

In [0]:
DELETE FROM item_demo WHERE i_category = 'Music';

### C1. Inspect the DELETE Operation Metrics

In [0]:
SELECT version, operation, operationMetrics
FROM (DESCRIBE HISTORY item_demo)
WHERE operation = 'DELETE';

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #00695c; font-size: 1.1em;">What to notice in operationMetrics</strong>
            <p style="margin: 8px 0 0 0; color: #333;"><code>numRemovedFiles = 0</code>, <code>numAddedFiles = 0</code>, <code>numCopiedRows = 0</code>, and <code>numDeletionVectorsAdded &gt; 0</code>. The data files are untouched; the deletion is recorded in a separate vector and applied at read time. On the V2 path (deletion vectors disabled) those numbers would be very different - parquet files rewritten, surviving rows copied across.</p>
        </div>
    </div>
</div>

## D. Create a Managed Iceberg Table and a Delta + UniForm Table

So far you have one table (<code>item_demo</code>) that is Delta with V3 reads enabled. To compare it against the other two interoperability options, CTAS two more copies of the same source data:

- <code>item_iceberg</code> - a <b>managed Iceberg</b> table (Iceberg-native, supports external read AND write)
- <code>item_uniform</code> - a <b>Delta + UniForm</b> table (primarily Delta, exposes Iceberg metadata for external readers)

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em;">Task 4.1 - CTAS a Managed Iceberg Table</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Create <code>item_iceberg</code> as a managed Iceberg table from the same source. Use <code>USING ICEBERG</code>.</p>
        </div>
    </div>
</div>

In [0]:
-- TODO: CTAS a managed Iceberg copy of samples.tpcds_sf1000.item
-- Hint: USING ICEBERG
CREATE OR REPLACE TABLE item_iceberg
<FILL_IN>
AS SELECT * FROM samples.tpcds_sf1000.item;

SHOW TBLPROPERTIES item_iceberg;

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Show Solution</strong>
      </div>
    </div>
  </summary>

<div class="code-block" data-language="sql">
CREATE OR REPLACE TABLE item_iceberg
USING ICEBERG
AS SELECT * FROM samples.tpcds_sf1000.item;
<br/>
SHOW TBLPROPERTIES item_iceberg;
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" id="prism-light" />
<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-tomorrow.min.css" rel="stylesheet" id="prism-dark" disabled />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="theme-toggle" style="position:absolute;top:8px;left:8px;padding:4px 10px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">&#x25D0;</button>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre class="code-pre" data-theme="light" style="background:#f8f8f8;border:1px solid #e0e0e0;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);

        var copyBtn = block.querySelector('.copy-btn');
        var toggleBtn = block.querySelector('.theme-toggle');

        copyBtn.onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = 'Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };

        toggleBtn.onclick = function() {
            var pre = block.querySelector('.code-pre');
            var isLight = pre.getAttribute('data-theme') === 'light';
            pre.setAttribute('data-theme', isLight ? 'dark' : 'light');
            pre.style.background = isLight ? '#1e1e1e' : '#f8f8f8';
            pre.style.border = isLight ? '1px solid #444' : '1px solid #e0e0e0';
            this.style.background = isLight ? '#333' : '#ddd';
            this.style.color      = isLight ? '#fff'  : '#333';
            this.style.border     = isLight ? '1px solid #666' : '1px solid #ccc';
            copyBtn.style.background = isLight ? '#333' : '#ddd';
            copyBtn.style.color      = isLight ? '#fff'  : '#333';
            copyBtn.style.border     = isLight ? '1px solid #666' : '1px solid #ccc';
            document.getElementById('prism-light').disabled = isLight;
            document.getElementById('prism-dark').disabled  = !isLight;
            Prism.highlightElement(codeEl);
        };
    });
})();
</script>

</details>

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em;">Task 4.2 - CTAS a Delta + UniForm Table</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Create <code>item_uniform</code> as a UC managed table with UniForm enabled at create time. Use <code>TBLPROPERTIES</code> with the V3 + universal format flags.</p>
        </div>
    </div>
</div>

In [0]:
-- TODO: CTAS a Delta + UniForm copy of samples.tpcds_sf1000.item
-- Hint: TBLPROPERTIES with delta.enableIcebergCompatV3 + delta.universalFormat.enabledFormats
CREATE OR REPLACE TABLE item_uniform
TBLPROPERTIES (
  FILL_IN
)
AS SELECT * FROM samples.tpcds_sf1000.item;

SHOW TBLPROPERTIES item_uniform;

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Show Solution</strong>
      </div>
    </div>
  </summary>

<div class="code-block" data-language="sql">
CREATE OR REPLACE TABLE item_uniform
TBLPROPERTIES (
  'delta.enableIcebergCompatV3'          = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
AS SELECT * FROM samples.tpcds_sf1000.item;
<br/>
SHOW TBLPROPERTIES item_uniform;
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" id="prism-light" />
<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-tomorrow.min.css" rel="stylesheet" id="prism-dark" disabled />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="theme-toggle" style="position:absolute;top:8px;left:8px;padding:4px 10px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">&#x25D0;</button>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre class="code-pre" data-theme="light" style="background:#f8f8f8;border:1px solid #e0e0e0;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);

        var copyBtn = block.querySelector('.copy-btn');
        var toggleBtn = block.querySelector('.theme-toggle');

        copyBtn.onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = 'Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };

        toggleBtn.onclick = function() {
            var pre = block.querySelector('.code-pre');
            var isLight = pre.getAttribute('data-theme') === 'light';
            pre.setAttribute('data-theme', isLight ? 'dark' : 'light');
            pre.style.background = isLight ? '#1e1e1e' : '#f8f8f8';
            pre.style.border = isLight ? '1px solid #444' : '1px solid #e0e0e0';
            this.style.background = isLight ? '#333' : '#ddd';
            this.style.color      = isLight ? '#fff'  : '#333';
            this.style.border     = isLight ? '1px solid #666' : '1px solid #ccc';
            copyBtn.style.background = isLight ? '#333' : '#ddd';
            copyBtn.style.color      = isLight ? '#fff'  : '#333';
            copyBtn.style.border     = isLight ? '1px solid #666' : '1px solid #ccc';
            document.getElementById('prism-light').disabled = isLight;
            document.getElementById('prism-dark').disabled  = !isLight;
            Prism.highlightElement(codeEl);
        };
    });
})();
</script>

</details>

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #00695c; font-size: 1.1em;">Format Comparison</strong>
            <p style="margin: 8px 0 0 0; color: #333;"><code>item_iceberg</code> properties are Iceberg-native (<code>format-version</code>, <code>write.parquet.compression-codec</code>) - no <code>delta.*</code> properties. <code>item_uniform</code> properties are mostly Delta with the UniForm + V3 flags layered on. Managed Iceberg supports both read AND write from external Iceberg engines; enabled-Delta variants (V3, UniForm) are read-only from outside Databricks.</p>
        </div>
    </div>
</div>

## E. External Access via PyIceberg

<p style="font-size: 1em; line-height: 1.6; color: #333">PyIceberg is the Python client for the Iceberg REST API. Pointing it at the Unity Catalog Iceberg REST endpoint demonstrates which UC tables external Iceberg clients can actually see.</p>

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em">Installing PyIceberg</strong>
            <p style="margin: 8px 0 0 0; color: #333"><strong>PyIceberg has already been installed</strong> by the lab classroom setup at the top of this notebook, so you can move straight to importing it. From a non-Databricks environment - a laptop, an EMR node, a CI runner - install it with pip:</p>
        </div>
    </div>
</div>

<div class="code-block">
%pip install pyiceberg[pyarrow]
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
    });
})();
</script>

In [0]:
%python
from pyiceberg.catalog import load_catalog

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)

warehouse = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
schema    = spark.sql("SELECT current_schema()  AS s").collect()[0]["s"]

# Note: pass connection options as keyword args only - omitting the first positional
# "name" argument prevents PyIceberg from reading any local ~/.pyiceberg.yaml config
# that might define a conflicting catalog (and try to use hive_metastore by default).
rest_catalog = load_catalog(
    type="rest",
    uri=f"https://{workspace_url}/api/2.1/unity-catalog/iceberg-rest",
    token=token,
    warehouse=warehouse,
)

print(f"Connected to UC Iceberg REST catalog: {warehouse}.{schema}")

### E1. Load Each Table Type

Load the three UC tables you have created or modified through PyIceberg. Loading returns the Iceberg metadata - column count, snapshot count - which proves the external Iceberg engine can see and parse the table.

In [0]:
%python
def load_table(table_name: str):
    fqn = f"{schema}.{table_name}"
    tbl = rest_catalog.load_table(fqn)
    n_cols = len(tbl.schema().fields)
    n_snapshots = len(tbl.metadata.snapshots)
    print(f"  OK  -> {fqn:40s} ({n_cols} cols, {n_snapshots} snapshot(s))")

print("Loading UC tables through the Iceberg REST catalog:")
print()
load_table("item_demo")     # Delta + Iceberg V3 reads (enabled in Part 2)
load_table("item_iceberg")  # native managed Iceberg
load_table("item_uniform")  # Delta + UniForm

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What this confirms</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Any UC table that exposes Iceberg metadata - V3 enabled, UniForm, or native Iceberg - is loadable from any Iceberg client. PyIceberg here is a stand-in for Snowflake, Trino, Flink, EMR, or any other Iceberg-aware engine.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Why we stopped at metadata</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Loading the table through PyIceberg returns the schema and snapshot count - that is the proof that an external Iceberg client can <b>see and parse</b> the table. Actually scanning the data with <code>tbl.scan().to_pandas()</code> requires PyIceberg to read the underlying parquet files directly from cloud storage (S3 / ADLS / GCS), which needs separate storage credentials that the Databricks notebook environment does not vend to PyIceberg by default. From a real external engine - Snowflake, Trino, EMR Spark, or PyIceberg running on a laptop with cloud credentials configured - the equivalent <code>SELECT * FROM item_iceberg LIMIT 10</code> works the same way it would against any other Iceberg table the engine knows about.</p>
        </div>
    </div>
</div>

## Lab Complete

You can now:

- Taken the Delta baseline `item_demo` through the **Iceberg V3** read-enable path
- Seen how V3 keeps deletion vectors and row tracking on, and how a `DELETE` records a deletion vector instead of rewriting parquet
- Created a **managed Iceberg** table and a **Delta + UniForm** table, and recognized each by its properties
- Used **PyIceberg** to verify which UC tables are visible from outside Databricks

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>